# GLP-1R计算流程复核
此Notebook使用真实归档AI结果重新计算结构指标并核验ESM，不重新调用AF3/MPNN/ESM。无训练、无湿实验标签。环境安装请先读README。

In [ ]:
from pathlib import Path
import sys, subprocess, csv, tempfile
root = Path.cwd().resolve()
if not (root / "predict.py").is_file():
    root = root.parent
assert (root / "predict.py").is_file(), "Open this notebook inside project or project/notebooks"
print("Project:", root)


In [ ]:
# New temporary output prevents overwriting the submitted example.
out = Path(tempfile.mkdtemp(prefix="notebook_review_", dir=root / "results"))
run = subprocess.run([sys.executable, "-B", str(root / "predict.py"), "--out", str(out)], cwd=root, text=True, encoding="utf-8", errors="replace", capture_output=True)
print(run.stdout)
if run.returncode: raise RuntimeError(run.stderr)


In [ ]:
with (out / "results.csv").open(encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))
for r in rows:
    print(r["group"], r["peptide_RMSD_median_A"], r["contact_retention_median_pct"], r["ESM_mean_masked_log_probability"], r["origin"])


In [ ]:
verify = subprocess.run([sys.executable, "-B", str(root / "tests/verify_regression.py"), "--results", str(out)], cwd=root, text=True, encoding="utf-8", errors="replace", capture_output=True)
print(verify.stdout)
if verify.returncode: raise RuntimeError(verify.stderr)


A30Q和Q23A为同优先级待验证候选，WT是对照，双突变是补充组合。15模型不是生物学重复，不能加显著性标记。